# Tidal Hydrodynamic Screening Model — A 2-Hour Workshop

Welcome! This notebook is a hands-on, gradual tour of an open-source software
model that estimates **how much energy we could harvest from ocean tides** —
specifically from the *currents* that flow between islands, not from dams or
barrages. No prior knowledge of oceanography, fluid dynamics, or numerical
modelling is assumed. We will build every concept up from scratch as we go.

## What is "tidal-current energy"?

The ocean rises and falls twice a day (the tide). Where that rising and falling
water is forced through a narrow gap — say, a strait between two islands — it
has to speed up to get the same volume of water through a smaller opening, just
like putting your thumb over a garden hose makes the water shoot out faster.
That fast-moving water carries **kinetic energy** (the energy of motion), and we
can extract some of it with underwater turbines that look and behave a lot like
wind turbines, only denser: seawater is ~800× denser than air, so the same speed
yields far more power.

The catch: before you can build anything, you need a map of **where** the currents
are strong enough to be worth it. That map is exactly what this model produces.

## What the model actually does

It is a **2D depth-averaged shallow-water solver**. Don't worry — that jargon is
explained piece by piece in the blocks below. In plain terms, the model:

1. Takes a map of the **seabed depth** (bathymetry) over a region of ocean.
2. Prescribes how the **tide rises and falls** at the edges of that region.
3. Uses the laws of physics to compute, over time, **how fast the water moves**
   at every point in between.
4. Converts those speeds into **power density** (watts per square metre), and
   averages over a full tidal cycle (~2 weeks) to get a fair estimate.
5. Highlights **hotspots** — places with enough power to be interesting.

By the end of this workshop you will understand what every piece of that
pipeline does, why it is built that way, and which knobs change the answer by
how much.

## Learning objectives

You will be able to:

1. Explain the staggered **Arakawa C-grid** and why it prevents a common
   numerical failure (the "checkerboard" pattern).
2. Build a model domain from bathymetry and a land mask.
3. Build tidal open-boundary forcing from harmonic constants.
4. Describe each term in the **shallow-water equations** and its effect on
   currents, in physical terms.
5. Choose a stable time step using the **CFL condition** — and explain why it
   exists.
6. Run the model, save output, and compute tidal power density.

## Agenda (≈ 120 minutes)

| Time  | Block | Topic |
|-------|-------|-------|
| 0:00 | **0** | Setup & orientation |
| 0:10 | **1** | Data pipeline & external datasets |
| 0:25 | **2** | The Arakawa C-grid |
| 0:40 | **3** | Tidal forcing |
| 0:55 | **4** | The shallow-water solver |
| 1:25 | **5** | CFL stability |
| 1:40 | **6** | Output, power density & validation |
| 1:55 | **7** | Wrap-up & further exercises |

⚡ Run cells top-to-bottom. Each individual simulation runs in **a few seconds**
on a typical laptop; the comparison sweeps take ~5–10 s. You can experiment
freely — change a number, rerun, see what happens. That is the best way to build
intuition.

## Requirements

```bash
pip install -r src/requirements.txt   # numpy, scipy, xarray, matplotlib, rasterio, fiona, netCDF4, pyyaml
```

No external datasets are required — all examples use synthetic grids we build
from scratch. Block 1 explains how to plug in real GEBCO / GOT4.10c data when
you have it for a production run.

---
## Block 0 — Setup & Orientation  ⏱ 0:00–0:10

Before we can simulate anything, we import the model's Python code and confirm
the supporting libraries are installed. The first code cell below also wires up
Python's import path so that this notebook runs correctly regardless of which
folder you launched Jupyter from — it walks up the directory tree until it finds
the project's `src/model/` folder and `README.md`.

If all of this is new to you: a Python **import** simply makes code defined in
another file available in the current notebook, like opening a toolbox so you
can reach for its tools. We import `numpy` (the standard library for fast array
math), `matplotlib` (for plotting), and the model's own modules (`StructuredGrid`,
`ShallowWaterSolver`, etc.), each of which we will explain when we first use it.

Run both cells below now. The first prints the project root and a success line;
the second ticks off the heavy libraries one by one.

In [ ]:
import sys, os
from pathlib import Path
import numpy as np

# Auto-locate the project root (contains src/model/ and README.md)
PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'docs' else Path.cwd().resolve()
for _ in range(5):
    if (PROJECT_ROOT / 'src' / 'model').is_dir() and (PROJECT_ROOT / 'README.md').is_file():
        break
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))
print('Project root:', PROJECT_ROOT)

from model.grid import StructuredGrid
from model.solver import ShallowWaterSolver, G, RHO_SEAWATER
from model.forcing import (
    ASTRO_FREQUENCIES, make_synthetic_tidal_boundary, build_tidal_boundary,
)
from model.bathymetry import regrid_bathymetry, elevation_to_depth
from model.utils import (
    coriolis, cfl_timestep, interpolate_to_u, interpolate_to_v,
    velocity_at_centres, speed, power_density,
)
from model.output import (
    create_results_dataset, write_netcdf, write_mean_power_geotiff,
    write_hotspots_geojson,
)
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 90

print('\u2713 model imported successfully')

In [ ]:
# Environment sanity check — quickly verify the heavy libs the notebook uses
import importlib
for mod in ['numpy','scipy','xarray','matplotlib','rasterio']:
    try:
        importlib.import_module(mod)
        print(f'  \u2713 {mod}')
    except Exception as e:
        print(f'  \u2717 {mod} -> {e}')
print(f'NumPy {np.__version__} | grid {StructuredGrid.__module__}')

Here is the 30,000-foot view of the whole pipeline. Don't worry about every
arrow yet — each block of this workshop unpacks one box:

```
 GEBCO bathymetry  ──┐
                    │   StructuredGrid ──┐
 GOT4.10c harmonics ┼─→  TidalBoundary   ├──→  ShallowWaterSolver ──┬──→ results.nc
                    │   (the ocean grid) │    (the physics engine)   ├──→ tidal_power_density.tif
 GADM shapefile    ──┘                   ┘                          └──→ hotspots.geojson
                                          ▲
                                          │
                        CFL condition picks the time step dt ──────┘
```

Reading it left-to-right:

- **Inputs (left)** are three kinds of real-world data: the shape of the seabed
  (bathymetry), the schedule of the tide (harmonic constants), and the outline
  of the land (a shapefile).
- **Assembly (middle)** packages those inputs into a `StructuredGrid` (a
  description of the ocean's geometry) and a `TidalBoundary` (a description of
  how the open ocean drives the model).
- **Engine (right)** is the `ShallowWaterSolver`, which turns that geometry and
  forcing into velocities and water levels over time.
- **Outputs (far right)** are three files: a full time history (`results.nc`), a
  map of mean power (`tidal_power_density.tif`), and a list of high-power
  locations (`hotspots.geojson`).

Now let's walk through each piece, starting with the data.

---
## Block 1 — Data Pipeline & External Datasets  ⏱ 0:10–0:25

A model is only as good as the data you feed it. This model consumes **three
categories** of external data. It is important to stress up front: *none of
them is required to run the notebook right now*. The model gracefully falls
back to synthetic (made-up) grids and forcing so that you can learn and test on
a clean slate. But when you want a *real* tidal resource assessment of an
actual coastline, you need all three.

| Dataset | Purpose | Source | Size |
|---------|---------|--------|------|
| **GEBCO 2024** | Seabed depth (bathymetry) — the shape of the ocean floor | gebco.net (free but licence acceptance) | ~2.7 GB global NetCDF |
| **GOT4.10c / FES2014 / TPXO9** | Tidal harmonic constants — the "schedule" of the tide at the open boundary | NASA / AVISO / OSU | 44 MB – 4 GB |
| **GADM + OSM shapefiles** | Land / coastline polygons — so the model knows which cells are ocean vs. island | gadm.org / Geofabrik | ~40 MB |

### Why bathymetry is the single most important input

Tides are surface gravity waves — the water surface goes up and down and that
disturbance travels across the ocean. The speed at which it travels depends on
water depth $h$ through $c = \sqrt{g\,h}$ (where $g \approx 9.81$ m/s² is
gravity): deeper water transmits the wave faster. Meanwhile, *friction* at the
seabed drags the water down — that drag is inversely proportional to depth,
scaling as $\tau_b \propto |U|\,U / h$. So depth controls **both** how fast the
tidal wave moves **and** how much energy it loses to the seabed.

The practical consequences are what make this an interesting problem:

- **Deep straits** (say 200 m, like San Bernardino Strait) transmit tidal energy
  efficiently and produce strong, clean currents.
- **Shallow shelves** dissipate most of the energy to friction before it goes
  anywhere useful.
- **Narrow passages** funnel the flow: the same volume of water must squeeze
  through a smaller cross-section, so it speeds up — the "thumb on the hose"
  effect again.

So the shape of the seabed literally shapes where the energy ends up. Getting
bathymetry right is therefore the foundation of everything that follows.

### The downloader helper

The project ships a `downloader.py` script to reduce the data-hunting tedium:

```bash
python downloader.py --all        # auto-fetch OSM, GADM, GOT4.10c
python downloader.py --gebco      # prints the manual GEBCO instructions
```

GEBCO itself can't be auto-downloaded because you must accept a licence on their
website, but the script walks you through it. After setup, your `data/`
directory holds the GEBCO NetCDF, the extracted GOT constituent files, and the
GADM/OSM shapefiles, all wired into the model via `config.yaml` (the run-time
configuration file). Block 0's environment cell already looked at `data/`; the
next cell explores what is actually on disk and inspects a GEBCO file if you
have one.

In [ ]:
# Take a look at what's actually on disk (graceful when nothing is present)
data_dir = PROJECT_ROOT / 'data'
print(f'Data directory: {data_dir}\n')
for p in sorted(data_dir.glob('*')) if data_dir.exists() else []:
    if p.is_dir():
        print(f'  \U0001f4c1 {p.name}/{len(list(p.glob("*")))} files')
    else:
        print(f'  \U0001f4c4 {p.name}  ({p.stat().st_size/1e6:.1f} MB)')

gebco_dir = data_dir / 'gebco_bathymetry'
nc_files = list(gebco_dir.glob('*.nc')) if gebco_dir.exists() else []
if nc_files:
    import xarray as xr
    ds = xr.open_dataset(nc_files[0], decode_times=False)
    print(f'\nGEBCO found: {nc_files[0].name}')
    print(f'  dims: {dict(ds.sizes)}  |  elevation: {float(ds.elevation.min()):.0f} to {float(ds.elevation.max()):.0f} m (positive up)')
    ds.close()
else:
    print('No GEBCO NetCDF found \u2014 the examples below use synthetic bathymetry instead.')

### The data-loading pipeline, step by step

The `bathymetry.py` module turns the raw GEBCO file into something the model can
use. Three functions do the heavy lifting, in order:

1. **`load_gebco(path, lon_min, lon_max, lat_min, lat_max)`** — GEBCO's global
   file covers the entire planet at ~450 m resolution, which is far more than we
   need for one region. This function *clips* it to your domain's bounding box
   (e.g. the Philippines, 116°–130°E and 4°–22°N) and returns 1D longitude and
   latitude arrays plus a 2D elevation grid. "Elevation" here follows the
   geophysics convention of **positive up**: land is +50 m, the seabed is
   −2000 m, mean sea level is 0.

2. **`regrid_bathymetry(lon, lat, elev, resolution_km)`** — 450 m is still far
   finer than the model's 1–5 km cells, and a finer grid means many more cells
   means a much slower simulation. This function *coarsens* the elevation to the
   target resolution by **bilinear interpolation** (basically averaging
   neighbouring points). The trade-off, which we'll quantify in Block 5, is that
   you lose detail — narrow channels may smear out — but you gain speed.

3. **`elevation_to_depth(elev)`** — the model uses the opposite sign convention
   to GEBCO: it wants **depth positive down** (seabed = +2000 m, land = 0).
   This function flips the sign and clips land to zero. Sign conventions are
   trivial but a source of endless bugs if you get them wrong, so the model
   isolates them in one place.

Finally, **`StructuredGrid.from_bathymetry()`** takes the cleaned depth array
and an optional land mask and assembles the full grid data structure. One very
convenient thing it does automatically: it marks every **wet** cell on the
domain's perimeter as an *open boundary* — these are the cells where the tide
will be prescribed (Block 3). The next cell demonstrates the regridding
visually on a synthetic field so you don't need GEBCO to follow along.

In [ ]:
# Demonstrate regridding on a synthetic high-res field (no GEBCO needed)
lon_hi = np.linspace(120, 122, 400)
lat_hi = np.linspace(10, 12, 400)
lon_g, lat_g = np.meshgrid(lon_hi, lat_hi)
elev_hi = -60.0 + 25.0 * np.sin(lon_g*8) * np.cos(lat_g*8)   # fake GEBCO, positive up

lon_new, lat_new, elev_new = regrid_bathymetry(lon_hi, lat_hi, elev_hi, resolution_km=2.0)
depth = elevation_to_depth(elev_new)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].pcolormesh(lon_hi, lat_hi, elev_hi, shading='auto', cmap='terrain'); axes[0].set_title(f'Input (GEBCO-like) {len(lon_hi)}\u00d7{len(lat_hi)}')
plt.colorbar(im0, ax=axes[0], label='elevation [m]')
im1 = axes[1].pcolormesh(lon_new, lat_new, depth, shading='auto', cmap='ocean'); axes[1].set_title(f'Regridded to ~2 km ({len(lon_new)}\u00d7{len(lat_new)})')
plt.colorbar(im1, ax=axes[1], label='depth [m]')
for ax in axes: ax.set_xlabel('lon'); ax.set_ylabel('lat')
plt.tight_layout(); plt.show()
print(f'Compression: {len(lon_hi)*len(lat_hi)//(len(lon_new)*len(lat_new))}\u00d7 fewer cells')

---
## Block 2 — The Arakawa C-Grid  ⏱ 0:25–0:40

To simulate the ocean, the model replaces continuous water with a **grid**: a
sheet of rectangular cells, like a spreadsheet. Each cell holds the values of
the variables at one location. The simplest arrangement would put *all*
variables at the centre of each cell — but that turns out to be numerically
unstable for the equations we're solving. Instead, this model uses a
**staggered Arakawa C-grid**, which spreads the variables across *different*
locations within each cell:

- **η (eta)** — the free-surface elevation, i.e. how much the water surface is
  raised or lowered relative to mean sea level. Stored at cell **centres**.
  Its array shape is `(ny, nx)` — there is one value per cell.
- **u** — the eastward velocity. Stored on the **east/west faces** of each cell,
  because that's the face water flows *through* when moving east-west. Its shape
  is `(ny, nx+1)` — there is one u-value per *vertical wall*.
- **v** — the northward velocity. Stored on the **north/south faces**. Its
  shape is `(ny+1, nx)` — one v-value per *horizontal wall*.

```
       v(j+1,i)            v(j+1,i+1)
          |                    |
    ------●--------------------●------      ● = v-point (north/south faces)
          |                    |
   u(j,i) *   η(j,i)   u(j,i+1 *  η(j,i+1)  * = u-point (east/west faces)
          |                    |
    ------●--------------------●------
          |                    |
       v(j,i)              v(j,i+1)
```

**Why stagger?** A pair of facts from numerical analysis make this worth the
bookkeeping:

1. **Natural pressure gradients.** The force that drives the water is the
   *slope* of the surface, $\partial\eta/\partial x$. With η at centres and u at
   faces, that gradient is just `(η_right − η_left) / dx` — a clean, second-order
   accurate difference with no extra interpolation. On a co-located grid you'd
   have to compute it awkwardly, and you'd lose accuracy.
2. **No checkerboard instability.** On a co-located grid, the standard
   finite-difference pressure gradient cancels out any "checkerboard" pattern
   (high−low−high−low across cells), so such patterns can grow unbounded and
   ruin the simulation. The C-grid naturally sees and damps them because the
   fluxes are computed *across* faces.

`StructuredGrid` is a Python *dataclass* (a tidy container of related fields)
with two builders:

- **`from_uniform()`** — a perfectly regular grid with constant spacing and a
  constant Coriolis parameter. Idealised, used for tests and teaching (and for
  most of this workshop).
- **`from_bathymetry()`** — a real domain built from a depth array and an
  optional land mask, with grid spacing derived from the lat/lon coordinates.

Let's create a small uniform grid and inspect its anatomy.

In [ ]:
g = StructuredGrid.from_uniform(nx=10, ny=8, dx=2000, dy=2000, lat0=5.0)   # 2 km grid at 5\u00b0N
print('Grid attributes (note the staggered shapes):')
print(f'  shape       = {g.shape}       (ny, nx) for \u03b7')
print(f'  dx, dy      = {g.dx:.0f}, {g.dy:.0f} m')
print(f'  h  shape    = {g.h.shape}       (depth at centres, positive down)')
print(f'  h_u shape   = {g.h_u.shape}   (depth at u-faces)')
print(f'  h_v shape   = {g.h_v.shape}   (depth at v-faces)')
print(f'  mask_u      = {g.mask_u.shape}   (wet mask at u-faces)')
print(f'  f  at 5\u00b0N = {g.f[0,0]:.3e} rad/s  (Coriolis, derived from latitude)')

In [ ]:
# Build a small domain with bathymetry + a land island, using from_bathymetry()
lon1 = np.linspace(120, 122, 40)
lat1 = np.linspace(10, 12, 32)
l2, t2 = np.meshgrid(lon1, lat1)
depth = np.full((32, 40), 50.0) + 10*np.sin(l2*4 - t2*3)
depth[10:18, 12:20] = 0.0        # dry island

land = depth < 2.0               # prevent overzealous masking: keep the island dry
dom = StructuredGrid.from_bathymetry(lon1, lat1, np.maximum(depth,2.0),
                                     land_mask=land, min_depth=2.0)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].pcolormesh(lon1, lat1, dom.h, shading='auto', cmap='ocean'); axes[0].set_title('Depth [m] (blue=ocean, 0=land)')
axes[0].scatter(dom.lon[dom.open_boundary], dom.lat[dom.open_boundary],
                c='red', s=3, label='open boundary')
axes[0].legend(markerscale=3, fontsize=8); plt.colorbar(im0, ax=axes[0])
im1 = axes[1].pcolormesh(lon1, lat1, dom.mask, shading='auto', cmap='Greens'); axes[1].set_title('Wet mask (1=wet, 0=dry)')
plt.colorbar(im1, ax=axes[1]); axes[0].set_xlabel('lon'); axes[0].set_ylabel('lat'); axes[1].set_xlabel('lon')
plt.tight_layout(); plt.show()
print(f'Wet cells: {dom.mask.sum()}/{dom.mask.size}  | Open-boundary cells: {dom.open_boundary.sum()}')

### Staggered-grid interpolation & the Coriolis force

Because η, u and v live at *different* points, the solver constantly needs to
move values between them. For example, the friction term in the u-equation needs
the water *depth* at the u-faces, but depth is stored at cell centres — so we
must interpolate. The `model/utils.py` module provides a small toolbox for
exactly this:

| Helper | What it does (plain English) |
|--------|------------------------------|
| `coriolis(lat)` | Computes the Coriolis parameter $f = 2\Omega\sin\phi$ (see below). |
| `interpolate_to_u(phi)` | Averages a cell-centre field $\phi$ to the **u-faces**: `result[:, i] = (phi[:, i-1] + phi[:, i]) / 2`. |
| `interpolate_to_v(phi)` | Same idea, but to the **v-faces** (averages vertically). |
| `velocity_at_centres(u,v)` | Averages the edge velocities u and v back to the **cell centres**, giving a single current vector per cell. |
| `speed(u,v)` | Computes $|U| = \sqrt{u_c^2 + v_c^2}$ at cell centres — the total current speed. |
| `power_density(u,v)` | Computes $P = \tfrac{1}{2}\,\rho\,|U|^3$ in W/m² — the quantity this whole model is built to estimate (Block 6). |

### What is the Coriolis force, and why do we care?

The Earth is a rotating sphere. From the perspective of something moving on the
surface (like a parcel of water), that rotation introduces a *fictitious* force
called the **Coriolis force** that bends moving objects. In the Northern
Hemisphere it bends them to the *right*; in the Southern Hemisphere, to the
*left*. The strength of the effect is summarised by a single number, the
**Coriolis parameter**:

$$f = 2\,\Omega\,\sin\phi$$

where $\Omega = 7.292\times10^{-5}$ rad/s is the Earth's rotation rate and
$\phi$ is latitude. At the equator ($\phi=0$), $\sin 0 = 0$ so there is **no**
Coriolis effect (water moves in a straight line). At the poles it is maximal.
The Philippines (4°–22°N) is in a **moderate** band — not negligible, but not
dominant either; it matters at scales of ~10 km and up, which is exactly our
model's resolution.

In the equations, Coriolis appears as `+ f·v` in the u-equation and
`− f·u` in the v-equation. Physically: if water is moving north (v>0), Coriolis
shoves it east (adds to u) — that's the "right turn" in the Northern Hemisphere.

The next cell lets you see the interpolation working on a tiny made-up field, and
shows how $f$ grows from zero at the equator to its full value at the pole.

In [ ]:
# Hands-on: see interpolation in action, and how Coriolis varies with latitude
phi = np.arange(6).reshape(2,3) + 10.0           # pretend cell-centre field
print('cell-centre field \u03c6:'); print(phi)
print('\ninterpolate_to_u  \u2192', interpolate_to_u(phi).shape, '(one extra column)')
print(interpolate_to_u(phi))
print('\nCoriolis f vs latitude (Philippines 4\u201322\u00b0N):')
for la in [0, 5, 10, 15, 20, 22, 90]:
    print(f'  lat={la:>3}\u00b0  f={coriolis(la):.3e} rad/s')

---
## Block 3 — Tidal Forcing  ⏱ 0:40–0:55

Gravity from the Moon and the Sun pulls the oceans back and forth, and because
the Earth rotates while the Moon orbits, this forcing is **periodic** — it
repeats at very precise frequencies. We exploit that regularity by decomposing
the tide into a handful of pure sine waves called **tidal constituents**, each
with a fixed period locked to an astronomical cycle:

$$\eta(t) = \sum_k A_k \cos(\omega_k\, t + \phi_k)$$

Each constituent $k$ has three ingredients:

- **$A_k$** — the *amplitude* (metres), how high the wave is at that location.
- **$\omega_k$** — the *angular frequency* (radians per second), how fast it
  oscillates. Period $T = 2\pi/\omega_k$.
- **$\phi_k$** — the *phase* (radians), a time offset saying "the peak happens
  this many seconds into the cycle here". Different locations peak at different
  times, which is what drives water from one place to another.

Just four constituents capture about 80–90% of the tidal variability worldwide,
and they are the ones the model uses by default:

| Constituent | Period (h) | Origin | Intuition |
|------------|-----------|--------|-----------|
| **M2** | 12.42 | Principal lunar *semi*diurnal | "Twice-a-day wobble from the Moon" — the dominant tide almost everywhere. |
| **S2** | 12.00 | Principal solar semidiurnal | The same effect from the Sun; slightly faster, so it slowly drifts in/out of phase with M2 (→ spring–neap cycle, next cell). |
| **K1** | 23.93 | Lunisolar *diurnal* | A once-a-day component; matters more in the tropics and Pacific. |
| **O1** | 25.82 | Principal lunar diurnal | Another daily component; combines with K1 to make tides feel "lopsided". |

### How the model gets $A_k$ and $\phi_k$

Amplitude and phase vary from place to place and are determined by global tide
models built from decades of satellite altimetry and tide gauges. The model
ships four readers that pull these constants out of different data formats:

- **`read_got_constituents()`** — GOT4.10c (NASA). Each constituent is one
  NetCDF file with amplitude in *centimetres* and phase in *degrees*. The reader
  converts to metres and radians for you. This is the recommended default —
  free, no registration, ~44 MB.
- **`read_fes_constituents()`** — FES2014 (AVISO/LEGOS). Higher resolution
  (1/16°, ~7 km) but requires AVISO registration and ~2 GB.
- **`read_tpxo_constituents()`** — TPXO9 (Oregon State). Stores amplitudes as
  real+imaginary parts ($A = \sqrt{\text{Re}^2+\text{Im}^2}$,
  $\phi = \arctan2(-\text{Im}, \text{Re})$). Single NetCDF, ~4 GB.
- **`make_synthetic_tidal_boundary()`** — for tests and teaching: uniform
  amplitude, zero phase at every boundary cell, no real data required. We will
  use this throughout the workshop.

All readers use `scipy.interpolate.RegularGridInterpolator` to interpolate the
global harmonic constants to *exactly the open-boundary cells* of your grid.

### "Open boundary" — what does that mean?

The model can only compute what happens *inside* its domain. At the edges ("open
boundaries"), instead of computing the tide, we **prescribe** it — we tell the
model "the surface here is this high at this time" using the harmonic formula
above. That prescribed motion is what sends waves into the domain and drives all
the currents. Think of it as shaking the edge of a tank to make waves inside.

Let's make a synthetic M2 boundary and watch the cosine wave it produces.

In [ ]:
# Make a synthetic M2 boundary and evaluate it at several times
bnd = make_synthetic_tidal_boundary(n_boundary_cells=1, amplitude=0.5, constituents=["M2"])
T_M2 = 2*np.pi/ASTRO_FREQUENCIES['M2']
print(f'M2 period = {T_M2/3600:.2f} h')

t = np.linspace(0, 2*T_M2, 400)
eta_m2 = bnd.evaluate(t)[:,0]
print(f'\u03b7(0)   = {eta_m2[0]:+.3f} m  (cos(0)=1)')
print(f'\u03b7(T/4) = {bnd.evaluate_at(T_M2/4)[0]:+.3f} m  (cos(\u03c0/2)\u22480)')

### The spring–neap cycle — why we run for two weeks

If we only added M2, the tide would be a perfectly regular cosine that never
changes size — the same amplitude forever. But the real ocean adds **S2**,
which is almost the same frequency but slightly faster (12.00 h vs 12.42 h).
Two nearby frequencies create a well-known phenomenon you've heard in sound
called *beating*: the two waves slide in and out of phase, and the combined
amplitude slowly swells and shrinks.

In tides this is the **spring–neap cycle**:

- **Spring tides** (~every 14.8 days): M2 and S2 are *in phase*, their highs add
  together → the largest tidal range → the **fastest currents** → the most
  power.
- **Neap tides** (halfway between): M2 and S2 are *out of phase*, the high of
  one cancels the low of the other → the smallest range → the **slowest
  currents** → the least power.

This has a huge practical consequence for resource assessment. If you ran the
model for only one day, you might land on a spring tide and *wildly
over-estimate* the average energy available, or on a neap tide and wildly
*under-estimate* it. To get a fair, representative number, you must simulate at
least one full spring–neap cycle — **about 15 days**. That is why
`config.yaml` defaults `duration_days: 15`.

Run the next cell to see this visually: a pure M2 wave (flat envelope) vs. the
swelling-and-shrinking envelope of M2+S2 over a month.

In [ ]:
bnd_m2s2 = make_synthetic_tidal_boundary(1, amplitude=0.5, constituents=['M2','S2'])
t_days = np.linspace(0, 30, 4000); t_s = t_days*86400
eta_m2   = bnd.evaluate(t_s)[:,0]
eta_m2s2 = bnd_m2s2.evaluate(t_s)[:,0]

fig, ax = plt.subplots(figsize=(11, 3.5))
ax.plot(t_days, eta_m2, lw=0.8, alpha=0.6, label='M2 only (constant envelope)')
ax.plot(t_days, eta_m2s2, lw=0.8, label='M2 + S2 (spring\u2013neap)')
ax.set(xlabel='time [days]', ylabel='\u03b7 [m]', title='Boundary elevation at one cell')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## Block 4 — The Shallow-Water Solver  ⏱ 0:55–1:25

This is the heart of the model — the "physics engine" that turns geometry and
forcing into currents. It implements the **depth-averaged shallow-water
equations**. That imposing name unpacks into three pieces:

- **Shallow-water** means the horizontal scale of the motion (tens to hundreds
  of km) is much larger than the water depth (tens to thousands of m). Under
  that assumption the waves are "long" compared to the depth, and we can treat
  the whole water column as moving together in the horizontal — we don't need
  to resolve the vertical structure.
- **Depth-averaged** is the formal way of saying "we collapse the vertical
  column into one layer". Instead of tracking velocity at every depth, we track
  a single $u$ and $v$ that represents the average speed of the whole column.
  This is an excellent approximation for tides and makes the problem 2D and
  fast.
- **Equations** are three coupled partial differential equations expressing
  conservation of **momentum** (Newton's $F=ma$ in two directions) and
  conservation of **mass** (water isn't created or destroyed).

### The equations, in symbols and in words

$$\underbrace{\frac{\partial u}{\partial t}}_{\text{acceleration}} =
\underbrace{-g\frac{\partial \eta}{\partial x}}_{\text{pressure gradient}}
\;+\;\underbrace{f\,v}_{\text{Coriolis}}
\;-\;\underbrace{\frac{C_d}{H}|U|\,u}_{\text{bottom friction}}
\;+\;\underbrace{A_h \nabla^2 u}_{\text{horizontal mixing}}$$

$$\frac{\partial v}{\partial t} = -g\frac{\partial \eta}{\partial y} - f\,u - \frac{C_d}{H}|U|\,v + A_h \nabla^2 v$$

$$\underbrace{\frac{\partial \eta}{\partial t}}_{\text{surface rises/falls}} =
-\underbrace{\frac{\partial}{\partial x}(H\,u)}_{\text{horizontal flux in x}}
\;-\;\underbrace{\frac{\partial}{\partial y}(H\,v)}_{\text{horizontal flux in y}}$$

where $H = h + \eta$ is the *total* water depth (seabed depth $h$ plus the
surface displacement $\eta$), $g = 9.81$ m/s², $\rho \approx 1025$ kg/m³, and
$C_d$, $A_h$ are tunable coefficients.

**In plain English:**

- **The x-momentum equation** says the eastward speed $u$ changes because: (a)
  water flows downhill — if the surface is higher to the west, the pressure
  gradient pushes water east; (b) the Coriolis force deflects northward-moving
  water eastward; (c) the seabed drags on the water, stealing momentum; (d)
  neighbouring cells smear velocity into each other through turbulence.
- **The y-momentum equation** is the same idea rotated 90°, with the Coriolis
  sign flipped (it deflects eastward-moving water southward — same "right turn"
  in the Northern Hemisphere, now happening to the v-component).
- **The continuity equation** is book-keeping for water: if more water flows
  *into* a cell than out of it (the fluxes don't balance), the surface must rise
  to hold the extra volume, and vice-versa. This is the equation that links the
  two velocity components back to the surface height and closes the system.

### The time-stepping scheme: forward–backward

The solver doesn't solve these equations all at once — it **marches forward in
small time steps $\Delta t$**. At each step it uses a clever ordering called
**forward-backward**:

1. **Apply the boundary condition.** Set $\eta$ at the open boundary cells to
   the tidal value prescribed by Block 3.
2. **Update the eastward velocity $u$** using the current $\eta$ (pressure
   gradient), current $v$ (Coriolis), and current speed (friction). This is the
   "forward" momentum step.
3. **Update the northward velocity $v$** — but using the *already-updated* $u$
   where Coriolis needs it. This is the "backward" trick, and it makes the
   scheme stable for the Coriolis coupling without needing tiny time steps.
4. **Update the surface $\eta$** from the divergence of the (new) fluxes —
   gravity then propagates that disturbance on the next step.

### Semi-implicit bottom friction (and why it matters)

The friction term is *non-linear* because it contains $|U|\,u$ — the speed times
the velocity. If you tried to add it explicitly (like the pressure gradient), the
simulation would go unstable in shallow water where friction is strong. Instead
the solver treats it **semi-implicitly**:

$$u^{n+1} = \frac{u^n + \Delta t \cdot (\text{all the other terms})}{1 + \Delta t\, \frac{C_d\,|U|}{H}}$$

That $1 + \ldots$ in the denominator is the trick. It guarantees friction can
only ever *reduce* the velocity, never reverse or amplify it, regardless of how
large $\Delta t$ is. This is what makes the model robust in shallow channels.

The first code cell below defines two helpers — one builds a simple 1D channel
domain (a rectangle that is open at both ends), the other creates an M2 tidal
boundary condition. We will reuse them for the rest of the workshop. Then we
run a small simulation and watch a tidal wave travel down the channel.

In [ ]:
# Helper that builds a 1-D tidal channel (closed walls, open boundaries at both ends)
def make_channel(L=50_000.0, H=30.0, nx=40, ny=3, lat0=0.0, coriolis_on=False):
    dx = L/nx
    g = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dx, lat0=lat0)
    g.h[:,:] = H; g.h_u[:] = H; g.h_v[:] = H
    g.mask[:] = True; g.mask_u[:] = True; g.mask_v[:] = True
    g.open_boundary[:] = False
    g.open_boundary[:,0] = True; g.open_boundary[:,-1] = True
    if not coriolis_on:
        g.f[:] = 0.0
    return g

def m2_eta_bc(g, amp=0.5):
    omega = ASTRO_FREQUENCIES['M2']
    def bc(t):
        e = np.zeros((g.ny, g.nx))            # NOTE: grid has no .eta; (ny,nx) is the \u03b7 shape
        e[g.open_boundary] = amp*np.cos(omega*t)
        return e
    return bc

print('Helpers defined. make_channel(40 cells) and m2_eta_bc() ready.')

In [ ]:
# Run a small M2-forced channel for 3 periods and watch the wave propagate
g_ch = make_channel(nx=30)
solver = ShallowWaterSolver(g_ch, cd=0.0025)
solver.set_open_boundary_eta(m2_eta_bc(g_ch))

T_M2 = 2*np.pi/ASTRO_FREQUENCIES['M2']
snaps = []
def cb(s, i):
    if i % 200 == 0:
        snaps.append((s.time, s.eta.copy(), s.u.copy()))
solver.run(dt=12.0, duration=2*T_M2, callback=cb, progress_interval=999999)
print(f'Ran {solver._step_count} steps in {len(snaps)} snapshots')

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
x_km = np.arange(g_ch.nx)*g_ch.dx/1000
for t_s, eta, _ in snaps:
    axes[0].plot(x_km, eta[1,:], lw=0.8, label=f't={t_s/3600:.1f}h')
axes[0].set(xlabel='x [km]', ylabel='\u03b7 [m]', title='\u03b7(x) snapshots along channel')
axes[0].legend(fontsize=7, ncol=2); axes[0].grid(alpha=0.3)
axes[1].plot([t/3600 for t,_,_ in snaps], [e[1,g_ch.nx//2] for _,e,_ in snaps], 'b.-', label='\u03b7 (mid)')
axes[1].plot([t/3600 for t,_,_ in snaps], [u[1,g_ch.nx//2] for _,_,u in snaps], 'r.-', label='u (mid)')
axes[1].set(xlabel='time [h]', title='Time series at midpoint'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

### 🛠 Exercise 1 — Vary the bottom drag (5 min)

**The setup.** Bottom friction is parameterised by a single dimensionless number
$C_d$ (the "drag coefficient"). It summarises how rough the seabed is:
smooth sand has $C_d \approx 0.002$, a rocky or coral bottom can be 0.003–0.005.
The friction force scales as $C_d\,|U|\,u / H$, so:

- **Larger $C_d$** → more drag → the water loses momentum faster to the seabed →
  **weaker currents**.
- Because power goes as the *cube* of speed ($P \propto U^3$, Block 6), even a
  modest drop in speed causes a *large* drop in power. Doubling the drag can
  easily halve the current, and halving the current cuts the power to
  **one-eighth**.

**Your task.** Before you run the cell, *predict* what you think will happen to
the peak speed as $C_d$ goes from 0 (frictionless!) up to 0.01 (very rough
seabed). Will the drop be linear? Faster than linear? Then run the cell and
compare. The cell sweeps five values of $C_d$, runs a 2-period simulation for
each, and prints the peak speed together with a rough estimate of power.

*(You do not need to edit anything to run it — just go. But to really learn,
change the list of `cd` values, or the channel depth `H`, and see how the
sensitivity changes.)*

In [ ]:
# Exercise 1: sweep bottom drag and report peak speed
print('Effect of bottom drag C_d on peak current speed:')
for cd in [0.0, 0.001, 0.0025, 0.005, 0.01]:
    s = ShallowWaterSolver(make_channel(nx=30), cd=cd)
    s.run(dt=12.0, duration=2*T_M2, progress_interval=999999)
    spd = float(np.max(speed(s.u, s.v)))
    print(f'  C_d = {cd:.4f}  ->  max|U| = {spd:.3f} m/s   (P ~ {0.5*1025*spd**3:.0f} W/m\u00b2)')

### Turning the physics knobs on and off

The momentum equation has several terms, and the solver lets you switch them
individually:

| Term | Knob | When it matters |
|------|------|-----------------|
| Pressure gradient $-g\,\partial\eta/\partial x$ | *always on* | Always. This is the primary driver of tidal flow — water accelerating "downhill". |
| Coriolis $f\,v$, $-f\,u$ | the grid's latitude `lat0` (set nonzero to turn on) | Wide domains (≥ 10 km); explains cross-channel slopes. |
| Bottom friction $-\frac{C_d}{H}|U|u$ | `cd` parameter | Dominant energy *sink* in shallow water; sets the steady current amplitude. |
| Horizontal mixing $A_h\nabla^2 u$ | `ah` parameter (0 = off) | Damps grid-scale noise; usually minor for well-resolved flows. |
| Non-linear advection $u\,\partial u/\partial x$ | `advection=True` | Narrow straits where velocity changes sharply over short distances. |

The next cell runs five variants of the same channel and reports the peak
$|u|$ and $|v|$ for each, so you can see *exactly* what each term buys you.
Look in particular at how Coriolis produces a non-zero $|v|$ — without rotation,
a purely east-west-forced channel would never flow north.

In [ ]:
def fast_run(grid, cd=0.0025, ah=0.0, advection=False, eta_bc=None):
    s = ShallowWaterSolver(grid, cd=cd, ah=ah, advection=advection)
    s.set_open_boundary_eta(eta_bc or m2_eta_bc(grid))
    s.run(dt=12.0, duration=2*T_M2, progress_interval=999999)
    return float(np.max(np.abs(s.u))), float(np.max(np.abs(s.v)))

base = make_channel(nx=30)
print('Linear baseline          : max|u|={:.3f}  max|v|={:.3f}'.format(*fast_run(base)))
print('With Coriolis (12\u00b0N)    : max|u|={:.3f}  max|v|={:.3f}  <- v develops from turning'.format(
      *fast_run(make_channel(nx=30, lat0=12.0, coriolis_on=True))))
print('With advection           : max|u|={:.3f}  max|v|={:.3f}'.format(*fast_run(base, advection=True)))
print('With eddy viscosity A_h=50: max|u|={:.3f}  max|v|={:.3f}'.format(*fast_run(base, ah=50.0)))
print('With extra drag C_d=0.01 : max|u|={:.3f}  max|v|={:.3f}'.format(*fast_run(base, cd=0.01)))

### Reading the comparison table — what each result tells us

- **Coriolis** spawns cross-channel flow: $|v|$ becomes non-zero, whereas it is
  essentially zero without rotation. This is the "right turn" in action. The
  Philippines (4°–22°N) sits in the moderate-Coriolis band, so it matters at our
  model's scales and is why Channel flow in the real ocean isn't perfectly
  one-dimensional.
- **Advection** has a small effect in this wide, uniform channel, but matters a
  great deal *in the real world* at narrow straits with sharp velocity
  gradients — exactly the places that are tidal-energy hotspots. There, the
  flow's own momentum carries it further than linear theory predicts.
- **Horizontal viscosity** ($A_h$) smoothes out seething small-scale wiggles.
  Setting it to 0 is fine *if* the grid resolves the flow; it becomes a useful
  safety net when the grid is coarse relative to the physics.
- **Friction** is the dominant **energy sink** in shallow water. It is what
  balances the pressure-gradient input at steady state, and it is where the
  tidal energy ultimately goes: into heat at the seabed. This is also the
  geophysical limit on how much we can extract with turbines — we're competing
  with the seabed for the same energy.

### Why straits are the hotspots — the funnel effect

This single idea is the key to the whole resource-assessment problem. Imagine
water flowing through a channel that narrows. Mass conservation says the
*volume* rate of flow $Q$ must be the same through every cross-section (water
isn't piling up). If the cross-sectional area $A$ shrinks but $Q$ is fixed,
then the *speed* must go up to compensate: $U \approx Q/A$. This is why putting
your thumb over the hose makes the jet faster.

The Philippine archipelago is dotted with inter-island channels — many narrow
gaps between large bodies of water that have different tidal phases. Those gaps
are where the current concentrates, and our screening model finds them. Combine
the funnel effect ($U$ goes up) with the cubic law ($P \propto U^3$ goes up
*a lot*), and you can see why a narrow strait at 3 m/s yields nearly *ten times*
the power density of an open bay at 1.5 m/s. That contrast is what the screening
map is built to reveal.

---
## Block 5 — CFL Stability  ⏱ 1:25–1:40

The solver is **explicit** — it computes the state at the next time step purely
from the state at the current step, with no solving of equations. Explicit
schemes are simple and fast, but they carry a strict speed limit called the
**Courant-Friedrichs-Lewy (CFL) condition**. The intuition is physical:

A tidal wave travels at speed $c = \sqrt{g\,h}$ (faster in deeper water). In one
time step $\Delta t$, information can travel a distance $c\,\Delta t$. The CFL
condition says **that distance must not exceed the grid spacing $\Delta x$** —
the wave must not be allowed to "jump over" a whole cell in a single step,
because then the grid would never "see" it and the simulation would lose the
physics (and go unstable). Formally:

$$\Delta t \;\leq\; s \cdot \frac{\Delta x_{\text{eff}}}{\sqrt{g\, h_{\max}}}, \qquad
\Delta x_{\text{eff}} = \left(\frac{1}{\Delta x^2} + \frac{1}{\Delta y^2}\right)^{-1/2}$$

where $s$ is a **safety factor** (default 0.5 — we run at half the theoretical
limit to absorb the non-linear terms and Coriolis, which the textbook formula
doesn't include). $h_{\max}$ is the *deepest* water in the domain, because that's
where the wave moves fastest and dictates the strictest limit.

### How $\Delta t$ scales — the cost of resolution

Read the formula carefully and you'll see two trends that govern the model's
cost:

- **Finer grid** (smaller $\Delta x$) → smaller allowed $\Delta t$. And because
  the total number of steps is $N = \text{duration}/\Delta t$, halving $\Delta x$
  doubles the number of *spatial* cells *and* roughly doubles the number of
  *time* steps — so the total work grows **roughly as $\Delta x^{-3}$**. A
  2× finer grid is ~8× more expensive. This is why screening models are coarse.
- **Deeper water** → faster waves → smaller $\Delta t$. A 5000 m deep ocean
  cell demands a much smaller time step than a 50 m shelf cell for the same
  grid.

If you violate the CFL condition, $\Delta t$ is too big and the model produces
**NaN** ("Not a Number") — the numerical equivalent of division-by-zero or
infinite overflow — and the run is garbage. The standard remedy is to lower the
`safety` factor (e.g. from 0.5 to 0.25), which buys margin at the cost of more
steps. The first cell below prints the CFL table for several resolutions and
depths so you can see these trends numerically; the second is a hands-on demo of
what a CFL violation does to the poor model.

The code branch that suffers most from this is real Philippine domains with deep
trenches (up to ~5000 m) — those dictate a small $\Delta t$ even though most of
the interesting dynamics are in the shallow straits. Production runs therefore
sometimes cap `max_depth` in the config to ease the CFL burden.

In [ ]:
# Maximum stable dt for combinations of resolution and depth (safety=0.5)
print('Max dt [s] by resolution (rows) x depth [m] (cols):\n')
depths = [10, 50, 200, 1000, 5000]
print(f'{"res":>8s}' + ''.join(f'{"h="+str(h)+"m":>10s}' for h in depths))
for res_km in [0.5, 1.0, 2.0, 5.0, 10.0]:
    dx = res_km*1000
    row = f'{str(res_km)+"km":>8s}'
    for h in depths:
        row += f'{cfl_timestep(dx,dx,h,safety=0.5):>10.1f}'
    print(row)
print('\nFiner grid or shallower h -> both -> smaller dt -> more steps -> slower run.')

### 🛠 Exercise 2 — Break the model (5 min)

This is the rare exercise where we *want* you to break something. The cell below
builds a 1 km grid, 50 m deep, and computes the CFL-safe time step (~1 s). It
then **deliberately runs with a 10× larger time step** to violate the
condition. Watch what happens: the surface elevation blows up into NaN.

**Your tasks:**

1. Run the cell as-is and confirm the "blew up" line. You have just witnessed a
   CFL violation — the classic failure mode of every explicit ocean model.
2. Find the line `dt_unsafe = dt_safe * 10` and change the `10` to `1` (i.e. use
   the safe time step). Rerun. The "Safe" line below it already shows what
   success looks like: a finite, sensible `max|η|` and no NaN. Now both halves
   of the cell should agree.
3. *Bonus:* try `5` instead of `10`. Sometimes the model survives 5× but the
   values are visibly wrong (enormous `max|η|`). The CFL limit is a *necessary*
   condition for stability, not a guarantee of accuracy — being close to the
   edge is risky even when you don't crash.

This exercise builds the muscle memory you need for production runs: whenever you
see NaN in the logs, **first** suspect the CFL condition, drop the safety factor
to 0.25 or set `dt` explicitly smaller, and rerun.

In [ ]:
# Exercise 2: CFL violation demo
gx = make_channel(L=20_000, H=50, nx=20)   # 1 km cells, 50 m deep
dt_safe = cfl_timestep(gx.dx, gx.dx, 50.0, safety=0.5)
print(f'Safe dt (CFL=0.5): {dt_safe:.2f} s')

dt_unsafe = dt_safe * 10
s = ShallowWaterSolver(gx, cd=0.0)
s.set_open_boundary_eta(m2_eta_bc(gx))
s.run(dt=dt_unsafe, duration=600, progress_interval=999999)
if np.any(np.isnan(s.eta)):
    print(f'Unsafe dt={dt_unsafe:.1f} s -> NaN! Model blew up \u2717')
else:
    print(f'Unsafe dt={dt_unsafe:.1f} s -> max|\u03b7|={np.max(np.abs(s.eta)):.2f} m (survived)')

# Stable re-run
s2 = ShallowWaterSolver(gx, cd=0.0); s2.set_open_boundary_eta(m2_eta_bc(gx))
s2.run(dt=dt_safe, duration=600, progress_interval=999999)
print(f'Safe  dt={dt_safe:.1f} s -> max|\u03b7|={np.max(np.abs(s2.eta)):.2f} m, no NaN \u2713')

---
## Block 6 — Output, Power Density & Validation  ⏱ 1:40–1:55

### The ½ρU³ law — where the power comes from

We've spent most of the workshop computing velocity. Now we convert that to the
thing we actually care about: **power**. A tidal-stream turbine is a kinetic
energy converter — it slows the water down and harvests the energy of motion.
The *instantaneous power per unit swept area* (think: the power available to a
1 m × 1 m hole in the water) is:

$$P = \tfrac{1}{2}\,\rho\,|U|^3 \quad [\text{W/m}^2]$$

with $\rho \approx 1025$ kg/m³ the density of seawater. Where does this come
from? Multiply three facts:

1. Kinetic energy *per unit volume* of moving water is $\tfrac{1}{2}\rho U^2$
   (the same $\tfrac{1}{2}mv^2$ you learned in school, divided by volume).
2. The *volume flux per unit area per unit time* is $U$ (a column of water $U$
   metres long passes through each square metre every second).
3. Multiply them: energy per volume × volume per area per time = power per area:
   $\tfrac{1}{2}\rho U^2 \times U = \tfrac{1}{2}\rho U^3$.

### Why the cube is everything

That exponent of **3** is the single most important fact in tidal resource
assessment. Because $P \propto U^3$:

| Speed $U$ (m/s) | Power $P$ (W/m²) | Relative to 1 m/s |
|----------------|------------------|--------------------|
| 1.0 | ~512 | 1× |
| 2.0 | ~4100 | **8×** |
| 3.0 | ~13 800 | **27×** |
| 4.0 | ~32 800 | **64×** |

**Doubling the current speed gives you eight times the power** — not twice. A
site at 2.5 m/s is not "a bit better" than a site at 2.0 m/s; it yields almost
double the power density. This is why *accurate velocity prediction* matters
more than anything else in this whole pipeline, and why every knob we explored
(C_d, depth, resolution, advection) matters — they all shift U, and U feeds
into a cubic. It is also why the screening model is deliberately *conservative*:
it slightly *under*-predicts currents (coarse grid, depth-averaging), so any
site it flags as promising is almost certainly worth a closer look.

### The three output files

The `output` module writes everything the downstream web map and GIS tools need:

- **`results.nc`** — a NetCDF containing the *full time series* of $\eta$, $u$,
  $v$ and the power density $P$ at every saved snapshot. This is the scientific
  output, useful for post-processing and re-analysis.
- **`tidal_power_density.tif`** — a Cloud-Optimised GeoTIFF of the **time-mean**
  power $\bar{P}(x,y)$, the map the web visualisation overlays. Time-averaging
  over a spring-neap cycle gives a *fair* representative number, rather than a
  single snapshot that might be a spring peak or a neap lull.
- **`hotspots.geojson`** — point features for every cell where $\bar{P}$
  exceeds a threshold (default 200 W/m²). These are the candidate sites — the
  shortlist for the next tier of modelling.

The first code cell below shows the cubic law numerically on a tiny synthetic
velocity field; the second walks through the full end-to-end output by faking a
short time series and writing all three files.

In [ ]:
# Instantaneous power density from a velocity field
# u lives on x-faces (ny, nx+1); v on y-faces (ny+1, nx). Centre speed = U.
U = np.array([0.5, 1.0, 1.5, 2.0, 2.5, 3.0])
ny, nx = 2, len(U)
u = np.zeros((ny, nx + 1))
u[:, :-1] = U          # left face of each cell carries the speed
u[:, 1:]  = U          # right face too -> centre speed = U exactly
v = np.zeros((ny + 1, nx))
P = power_density(u, v, rho=1025.0)
print('speed |U| [m/s]:', np.round(speed(u, v).ravel(), 2))
print('power  P [W/m2]:', np.round(P.ravel(), 1))
print()
print('Cubic law: doubling 1 -> 2 m/s is NOT 2x power, it is 8x (', round((2/1)**3), 'x).')


In [ ]:
# End-to-end output: build a degree-grid domain, fake a time series, and write all three outputs
import tempfile, json
lon1 = np.linspace(120, 122, 10); lat1 = np.linspace(10, 12, 8)
g_out = StructuredGrid.from_bathymetry(lon1, lat1, np.full((8,10), 50.0), min_depth=2.0)
n_t = 24; times = np.arange(n_t)*3600.0
eta = (0.5*np.sin(2*np.pi*times[:,None,None]/T_M2)) * np.ones((n_t, 8, 10))
u   = (0.3*np.cos(2*np.pi*times[:,None,None]/T_M2)) * np.ones((n_t, 8, 11))
v   = (0.1*np.sin(2*np.pi*times[:,None,None]/T_M2)) * np.ones((n_t, 9, 10))
spd  = np.sqrt(u[:,:,1:]**2 + v[:,1:,:]**2)
Pwr  = 0.5*1025*spd**3
Pmean = Pwr.mean(0)
ds = create_results_dataset(g_out, times, eta, u, v, Pwr)
tmp = tempfile.mkdtemp()
write_netcdf(ds, os.path.join(tmp, 'results.nc'))
write_mean_power_geotiff(g_out, Pmean, os.path.join(tmp, 'tidal_power_density.tif'))
gj_path = os.path.join(tmp, 'hotspots.geojson')
write_hotspots_geojson(g_out, Pmean, threshold=5.0, path=gj_path)
gj = json.load(open(gj_path))
print(f'Wrote results.nc, tidal_power_density.tif, and {len(gj["features"])} hotspot feature(s) to {tmp}')
for f in gj['features'][:3]:
    print(f"  lon={f['geometry']['coordinates'][0]:.2f} lat={f['geometry']['coordinates'][1]:.2f} "
          f"P={f['properties']['power_density_Wm2']:.1f} W/m2 h={f['properties']['depth_m']:.0f} m")

### Validation — how do we know the solver is correct?

Any model can produce numbers; the question is whether they're *right*. The
project's test suite checks the solver against analytical solutions — exact
answers from textbook physics that we can compare to. Below you can run two of
them in seconds:

**1. The Merian seiche — a standing wave in a closed bathtub.**

If you tilt the water in a closed rectangular basin and let it go, it sloshes
back and forth as a "standing wave" (a *seiche*). The fundamental period is
given by **Merian's formula**:

$$T = \frac{2 L}{\sqrt{g\, h}}$$

where $L$ is the basin length and $h$ is its depth. This is pure physics — no
tuning, no parameters. If our solver's wave propagation is correct, the model's
oscillation period must match this formula. The cell below builds a closed
basin with a cosine initial tilt, runs it, and recovers the period from the
zero-crossings of the surface height. The error should be well under 10%.

**2. Mass conservation — water is neither created nor destroyed.**

In a closed basin with no flow through the walls, the total volume of water
is a constant — it just redistributes. The solver's `total_volume()` method
computes $\sum (h + \eta)\,\Delta x\,\Delta y$ over all cells; we compare its
value before and after a run. Production runs keep this drift **below 0.01%**.
If it were much larger, that would be a bug — the solver would be silently
adding or removing water. The cell below runs a Gaussian "bump" in a closed
basin and reports the drift.

These two tests are the bedrock of our confidence: *mass is conserved*, and
*waves move at the right speed*. Everything more elaborate (real Philippine
bathymetry, GOT forcing, power maps) is built on top of this same solver, so if
these simple test cases pass, the engine is sound.

In [ ]:
# Seiche: closed basin, initial cos(pi x/L) tilt, compare to Merian's formula
L, H, nx, ny = 10_000.0, 10.0, 40, 3
dx = L/nx
gs = StructuredGrid.from_uniform(nx=nx, ny=ny, dx=dx, dy=dx, lat0=0.0)
gs.h[:,:] = H; gs.h_u[:] = H; gs.h_v[:] = H
gs.mask[:] = True; gs.mask_u[:] = True; gs.mask_v[:] = True
gs.open_boundary[:] = False; gs.f[:] = 0.0            # closed, no rotation
T_expected = 2*L/np.sqrt(G*H)
x = np.arange(nx)*dx + dx/2
se = ShallowWaterSolver(gs, cd=0.0)
se.set_initial_conditions(eta0=np.tile(0.1*np.cos(np.pi*x/L), (ny,1)))

traj = []
for k in range(int(2*T_expected/1.0)):                # dt=1 s, 2 periods
    se.step(1.0)
    if k % 20 == 0:
        traj.append(se.eta[1, nx//2])
# crude period estimate from zero crossings
traj = np.array(traj)
zc = np.where(np.diff(np.sign(traj)) != 0)[0]
dt_traj = 20.0
T_measured = (zc[2]-zc[0])*dt_traj if len(zc) >= 3 else float('nan')
print(f'Merian expected T = {T_expected:.1f} s')
print(f'Measured        T = {T_measured:.1f} s  (error {100*abs(T_measured-T_expected)/T_expected:.1f}%, pass < 10%)')

fig, ax = plt.subplots(figsize=(8, 2.6))
ax.plot(np.arange(len(traj))*dt_traj/60, traj, lw=1)
ax.set(xlabel='time [min]', ylabel='\u03b7 @ mid [m]', title='Seiche oscillation')
ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

In [ ]:
# Mass conservation: closed 20x20 basin, Gaussian bump, measure drift over 30 min
gm = StructuredGrid.from_uniform(nx=20, ny=20, dx=1000, dy=1000, lat0=0.0)
gm.h[:,:] = 40.0; gm.h_u[:] = 40.0; gm.h_v[:] = 40.0
gm.mask[:] = True; gm.mask_u[:] = True; gm.mask_v[:] = True
gm.open_boundary[:] = False; gm.f[:] = 0.0
xc = np.arange(20)*1000+500; yc = np.arange(20)*1000+500
yy, xx = np.meshgrid(yc, xc, indexing='ij')
eta0 = 0.5*np.exp(-((xx-10000)**2+(yy-10000)**2)/(2*3000**2))

s = ShallowWaterSolver(gm, cd=0.0)
s.set_initial_conditions(eta0=eta0)
V0 = s.total_volume()
s.run(dt=2.0, duration=1800, progress_interval=999999)
V1 = s.total_volume()
drift = 100*abs(V1-V0)/V0
print(f'Initial volume = {V0:.3e} m\u00b3')
print(f'Final   volume = {V1:.3e} m\u00b3')
print(f'Mass drift      = {drift:.4f}%  (pass < 0.01%) \u2713' if drift < 0.01 else f'drift={drift:.4f}% (too large) \u2717')

---
## Block 7 — Wrap-up & Further Exercises  ⏱ 1:55–2:00

### The complete pipeline (what `run.py` does)

You've now met every piece individually. Here is how the production entry point
`run.py` chains them together — the same pipeline the web map is built on:

```
GEBCO  -> load_gebco -> regrid -> elevation_to_depth -\
                                       .------------->|
GADM .shp -> build_land_mask    -----'              |
                                                    v
                      StructuredGrid --> ShallowWaterSolver
GOT/FES/TPXO -> read_constituents -> TidalBoundary -->  |
                                                dt from CFL  |
                                                       run()  --+--> results.nc
                                                                +--> tidal_power_density.tif
                                                                +--> hotspots.geojson
```

To run the real thing on the Philippine domain (after sourcing the data):

```bash
python downloader.py --all          # fetch OSM, GADM, GOT4.10c
# (manually place GEBCO_2024.nc into data/gebco_bathymetry/)
python -m src.model.run            # uses config.yaml (got forcing, 15 days)
docker compose up -d               # web map at http://localhost:5000
```

### Key takeaways — the cheat sheet

| Knob | What it controls | How the answer changes |
|------|-------------------|------------------------|
| **Bathymetry (depth $h$)** | Wave speed ($c=\sqrt{gh}$) and friction ($\propto 1/h$). | The single biggest input. Deep straits concentrate energy; shallow shelves kill it. |
| **Grid resolution ($\Delta x$)** | The smallest feature the model can "see". | Finer grids capture narrow straits but cost ~$\Delta x^{-3}$ more compute. Doubling resolution ≈ 8× slower. |
| **Bottom drag ($C_d$)** | How much momentum the seabed steals. | Higher $C_d$ → slower currents → *much* less power (roughly $P \propto$ power falls as $C_d$ rises). |
| **Tidal amplitude ($A$)** | How hard the boundary is being "shaken". | $A$ drives $U$ drives $P$; cubic sensitivity. Small errors in forcing = large errors in resource. |
| **Coriolis (latitude)** | The rightward (NH) deflection of moving water. | Matters at scales ≥ 10 km; generates cross-channel flow. Philippines is in the moderate band. |
| **Advection (on/off)** | Non-linear momentum transport by the flow itself. | Small in open water; large in narrow straits with sharp velocity gradients — i.e. the hotspots. |
| **CFL safety ($s$)** | How close to the stability edge we run. | Lower (0.25) is safer in complex bathymetry; costs more steps. If you see NaN, lower this first. |

### 🛠 Take-home exercises

These build on what you've learned and push a little further. Bring your
solutions to the next session!

1. **Channel depth sweep.** Rerun Exercise 1's sweep but vary the depth
   `H = [10, 30, 100, 300]` instead of `cd`. Plot peak speed vs depth. Does it
   increase forever, or saturate? Why? *(Hint: deeper water has less friction
   per unit volume, but also transmits the wave differently — there's a
   competition.)*
2. **Funnel effect.** Build a channel whose middle half is narrower (fewer rows
   `ny`) and confirm the velocity speeds up where the cross-section shrinks.
   Compare the speed-up ratio to the area ratio $A_1/A_2$.
3. **Spring–neap in the channel.** Replace `m2_eta_bc` with an M2+S2 boundary
   and run for 15 days; compute the time-mean **and** the P95 (95th-percentile)
   power density. Why is P95 a more useful design number than the mean for a
   turbine engineer?
4. **Real data.** Download GOT4.10c (`python downloader.py --all`) and run
   `python -m src.model.run` over the Philippine domain. Inspect
   `output/hotspots.geojson` and see if you can locate San Bernardino Strait —
   one of the world's strongest tidal currents.

### Where to read more

- **`docs/MODEL.md`** — the full physics & methodology reference. The math here
  is expanded, with proper derivations and citations.
- **`src/notebooks/01_hydrodynamic_model.ipynb`** — a first-principles
  walkthrough that starts from "what are tides, really?" with more analogies.
- **`README.md`** — the quick start, the full configuration table, and a
  troubleshooting guide for when the model misbehaves.

---
### 🎉 You finished the workshop

You now know how the screening model turns bathymetry and tidal harmonics into a
tidal-stream power map, and **why** every parameter changes the answer by the
amount it does. You've seen the cubic law in action, broken the model with a CFL
violation (and fixed it), verified mass conservation, and recovered Merian's
formula numerically. For real sites that look promising, the next tier is to
refine them with an unstructured-mesh model like **TELEMAC-2D** — but that's a
workshop for another day.